In [44]:
import pandas as pd
import requests
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpacthes
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("2")
print(f"pandas version :{pd.__version__}")
print(f"requests version :{requests.__version__}")
print(f"loaded at :{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

2
pandas version :2.2.2
requests version :2.32.4
loaded at :2026-05-28 11:17:45


In [45]:
raw_df=pd.read_csv('/content/drive/MyDrive/messy_sales_data.csv')
print(f"rowa {raw_df.shape[0]}")
print(f"col {raw_df.shape[1]}")
print(f"columun names :{raw_df.columns.tolist()}")
print(raw_df.head())

rowa 30
col 9
columun names :['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']
   order_id customer_name   product     category  quantity  unit_price  \
0      1001  Ramesh Kumar    Laptop  Electronics       2.0       45000   
1      1002    Priya Nair       NaN  Electronics       1.0       15000   
2      1003    AMIT VERMA  Keyboard  Accessories       3.0        1200   
3      1004  Sunita Patel   Monitor  Electronics       NaN       22000   
4      1005  Ramesh Kumar    Laptop  Electronics       2.0       45000   

   order_date       city    sales_rep  
0  2024-01-05     Mumbai  Anil Sharma  
1  2024-01-07      Delhi   Sunita Rao  
2  2024-01-08  Bangalore  Anil Sharma  
3  2024-01-10    Chennai   Ravi Kumar  
4  2024-01-05     Mumbai  Anil Sharma  


In [46]:
print("\n[1] MISSING VALUES per column:")
print(raw_df.isnull().sum())
print(f"\n[2] DUplicate rows:{raw_df.duplicated().sum()}")
print("3")
print(raw_df.dtypes)
print("\n[4] uni",raw_df['category'].dropna().unique().tolist())
print("[4] orderd d",raw_df['order_date'].unique()[:8].tolist())
print("[4] sample ",raw_df['customer_name'].dropna().unique()[:8].tolist())



[1] MISSING VALUES per column:
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] DUplicate rows:0
3
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] uni ['Electronics', 'Accessories']
[4] orderd d ['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '07-01-2024', '2024-01-12', '2024-01-13', '2024-01-15']
[4] sample  ['Ramesh Kumar', 'Priya Nair', 'AMIT VERMA', 'Sunita Patel', 'kiran mehta', 'Deepak Singh', 'Ananya Das', 'Vikram Iyer']


In [47]:
df=raw_df.copy()
print(f"shape {df.shape}")
print("df=raw_df.copy")

shape (30, 9)
df=raw_df.copy


In [48]:
print(f"{raw_df.isnull().sum().sum()}")
df['customer_name'].fillna('unknown Customer', inplace=True)
m=df['quantity'].median()
df['quantity'].fillna(m,inplace=True)
print(f"{df.isnull().sum().sum()}")
df['category'].fillna('uncategorised',inplace=True)
df['product'].fillna('notdefined',inplace=True)
print(f"Total Duplicated values after fix: {df.duplicated().sum()}")

7
2
Total Duplicated values after fix: 0


In [49]:
#Before Duplication
print(f"Rows before duplication:{len(df)}")
print(f"Duplicates rows found :{df.duplicated().sum()}")
#Removal of duplication
#keep = False to marks ALL copies of duplicate as True
duplicate_mask = df.duplicated(keep=False)
print("\nThe duplicate rows(all copies shown):")
print(df[duplicate_mask][['order_id','customer_name','product','order_date']].to_string(index=False))
df.drop_duplicates(inplace=True)
df.reset_index(drop=True,inplace=True)
print(f"\nRows after duplication:{len(df)}")
print(f"Rows removed:{len(df[duplicate_mask])}")

Rows before duplication:30
Duplicates rows found :0

The duplicate rows(all copies shown):
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []

Rows after duplication:30
Rows removed:0


In [50]:
print(df['order_date'].head(10).tolist())

['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13', '2024-01-15', '2024-01-15']


In [74]:
df['order_date']=pd.to_datetime(df['order_date'],dayfirst=True,errors='coerce')

nm=df['order_date'].isnull()

df.loc[nm,'order_date'] = pd.to_datetime(raw_df.loc[nm,'order_date'], dayfirst=False, errors='coerce')

# Identify remaining null dates after trying both dayfirst options
remaining_nulls_mask = df['order_date'].isnull()
nc=remaining_nulls_mask.sum()
print(f"Remaining unparseable dates: {nc}")

# Fill remaining NaT values with the median of the valid dates
if nc > 0:
    median_date = df['order_date'].median()
    df.loc[remaining_nulls_mask, 'order_date'] = median_date
    print(f"Filled {nc} unparseable dates with median date: {median_date.strftime('%Y-%m-%d')}")

# Re-derive date components after handling all nulls
df['year']=df['order_date'].dt.year
df['month']=df['order_date'].dt.month
df['month_name']=df['order_date'].dt.strftime('%B')
df['day_name']=df['order_date'].dt.strftime('%A')
print(df[['order_date','year','month','month_name','day_name']].head(10).to_string(index=False))

Remaining unparseable dates: 1
Filled 1 unparseable dates with median date: 2024-02-20
order_date  year  month month_name  day_name
2024-05-01  2024      5        May Wednesday
2024-07-01  2024      7       July    Monday
2024-08-01  2024      8     August  Thursday
2024-10-01  2024     10    October   Tuesday
2024-05-01  2024      5        May Wednesday
2024-07-01  2024      7       July    Monday
2024-12-01  2024     12   December    Sunday
2024-01-13  2024      1    January  Saturday
2024-01-15  2024      1    January    Monday
2024-01-15  2024      1    January    Monday


In [52]:
#Standardizing names
print("Names before standardization:")
print(df['customer_name'].unique().tolist())

#Notice "Amit Vera","KIRAN MERA",ramesh kumar" are inconsistent

df['customer_name'] = df['customer_name'].str.strip().str.title()
print("\nNames after standardization:")
print(df['customer_name'].unique().tolist())

Names before standardization:
['Ramesh Kumar', 'Priya Nair', 'AMIT VERMA', 'Sunita Patel', 'kiran mehta', 'Deepak Singh', 'unknown Customer', 'Ananya Das', 'Vikram Iyer', 'Pooja Gupta', 'SURESH RAO', 'Meera Joshi', 'Arjun Nair', 'Tanvi Mehta', 'Kiran Mehta', 'Rohit Verma', 'Sneha Reddy', 'Gaurav Shukla', 'Nisha Kapoor', 'Ajay Tiwari', 'ANANYA DAS', 'Preeti Saxena', 'Amit Bose', 'Rekha Nair', 'Harish Pillai', 'Sanjay Dubey', 'Kavya Nambiar']

Names after standardization:
['Ramesh Kumar', 'Priya Nair', 'Amit Verma', 'Sunita Patel', 'Kiran Mehta', 'Deepak Singh', 'Unknown Customer', 'Ananya Das', 'Vikram Iyer', 'Pooja Gupta', 'Suresh Rao', 'Meera Joshi', 'Arjun Nair', 'Tanvi Mehta', 'Rohit Verma', 'Sneha Reddy', 'Gaurav Shukla', 'Nisha Kapoor', 'Ajay Tiwari', 'Preeti Saxena', 'Amit Bose', 'Rekha Nair', 'Harish Pillai', 'Sanjay Dubey', 'Kavya Nambiar']


In [53]:
#FIXING THE CATEGORIES AND PRODUCT FOR A PERFECT MATCH
#data shows keyboard labelled as 'Electronics' but it needed to change 'Accessories'

#Step1 BUILD A BOOLEAN MASK -- TRUE for rows where both conditions are true
wrong_mask = (df['product'] == 'Keyboard') & (df['category'] == 'Electronics')

print(f"Rows to fix: {wrong_mask.sum()}")
print(f"Before fix")
print(df[wrong_mask][['order_id','product','category']].to_string(index=False))

#Step2 USE .loc to update only the rows
#df.loc [row_condition,'column'] = new_value
df.loc[wrong_mask,'category'] = 'Accessories'

print("After Fix")
print(df[wrong_mask][['order_id','product','category']].to_string(index=False))
print("All unique categories now:",sorted(df['category'].unique().tolist()))

Rows to fix: 1
Before fix
 order_id  product    category
     1024 Keyboard Electronics
After Fix
 order_id  product    category
     1024 Keyboard Accessories
All unique categories now: ['Accessories', 'Electronics', 'uncategorised']


In [54]:
df['quantity']=pd.to_numeric(df['quantity'],errors='coerce').astype(int)
df['unit_price']=pd.to_numeric(df['unit_price'],errors='coerce')
df['revenue']=df['quantity']*df['unit_price']
print(df[['customer_name','product','quantity','unit_price','revenue']].head(8).to_string(index=False))



   customer_name    product  quantity  unit_price  revenue
    Ramesh Kumar     Laptop         2       45000    90000
      Priya Nair notdefined         1       15000    15000
      Amit Verma   Keyboard         3        1200     3600
    Sunita Patel    Monitor         2       22000    44000
    Ramesh Kumar     Laptop         2       45000    90000
     Kiran Mehta      Mouse        10         800     8000
    Deepak Singh Headphones         2        3500     7000
Unknown Customer     Webcam         1        2500     2500


In [73]:
missing_count   = df.isnull().sum().sum()
duplicate_count = df.duplicated().sum()
date_nulls      = df['order_date'].isnull().sum()
revenue_nulls   = df['revenue'].isnull().sum()


checks_passed   = sum([
    missing_count   == 0,
    duplicate_count == 0,
    date_nulls      == 0,
    revenue_nulls   == 0,
    len(df)         > 0
])
quality_score = checks_passed * 20


print("=" * 55)
print("  POST-CLEANING VALIDATION REPORT")
print("=" * 55)
print(f"  Original rows   : {len(raw_df)}")
print(f"  Cleaned rows    : {len(df)}")
print(f"  Rows removed    : {len(raw_df) - len(df)} (duplicates)")
print(f"  Missing values  : {missing_count}")
print(f"  Duplicates      : {duplicate_count}")
print(f"  Date nulls      : {date_nulls}")
print(f"  Revenue nulls   : {revenue_nulls}")
print(f"  Columns total   : {len(df.columns)}")
print("=" * 55)
print(f"  DATA QUALITY SCORE : {quality_score}/100")
print(f"  DATA IS CLEAN      : {quality_score == 100}")
print("=" * 55)


if missing_count > 0:
    print("\n  ACTION REQUIRED: Missing values detected.")
    print("  → Use df['column'].fillna(value, inplace=True)")
    print("  → For numbers: fillna(df['column'].median())")
    print("  → For text   : fillna('Unknown')")


if duplicate_count > 0:
    print("\n  ACTION REQUIRED: Duplicate rows detected.")
    print("  → Use df.drop_duplicates(inplace=True)")


if date_nulls > 0:
    print("\n  ACTION REQUIRED: Unparseable dates found.")
    print("  → Check for unusual date formats in the raw data")
    print("  → Use pd.to_datetime(col, dayfirst=True, errors='coerce')")


if quality_score == 100:
    print("\n  All checks passed. Data is ready for analysis.")

  POST-CLEANING VALIDATION REPORT
  Original rows   : 30
  Cleaned rows    : 30
  Rows removed    : 0 (duplicates)
  Missing values  : 80
  Duplicates      : 0
  Date nulls      : 16
  Revenue nulls   : 0
  Columns total   : 14
  DATA QUALITY SCORE : 60/100
  DATA IS CLEAN      : False

  ACTION REQUIRED: Missing values detected.
  → Use df['column'].fillna(value, inplace=True)
  → For numbers: fillna(df['column'].median())
  → For text   : fillna('Unknown')

  ACTION REQUIRED: Unparseable dates found.
  → Check for unusual date formats in the raw data
  → Use pd.to_datetime(col, dayfirst=True, errors='coerce')


In [56]:
output_filename = 'clean_data.csv'
df.to_csv(output_filename,index=False)
print(f"Cleaned data saved to {output_filename}")
print(f"final dataset's shape {df.shape}")

Cleaned data saved to clean_data.csv
final dataset's shape (30, 14)


In [57]:
SERP_URL = 'https://serpapi.com/search.json'
SERP_API_KEY = 'f3ed573e865679da54dbfae8a1919525f0d91ba7f2a78858579c8d6b5441e286' # Please replace with your actual SerpAPI key
SEARCH_QUERY = 'Data Engineer India'
print(f"SerpAPI Key  : {'Set (live data)' if SERP_API_KEY != 'YOUR_SERPAPI_KEY_HERE' else 'Not set (fallback data will be used)'}")
print(f"Search query: {SEARCH_QUERY}")

SerpAPI Key  : Set (live data)
Search query: Data Engineer India


In [58]:
# ============================================================
# CELL 22 — EXTRACT: Fetch Job Listings from SerpAPI
# ============================================================


def fetch_jobs(query, api_key, num_pages=2):
    """
    Fetches job listings from Google Jobs via SerpAPI.


    Parameters:
        query    (str) : The job search query (e.g. 'Data Engineer India')
        api_key  (str) : Your SerpAPI key
        num_pages(int) : Number of result pages to fetch (default: 2)


    Returns:
        list : A list of job dictionaries
    """
    all_jobs = []


    for page in range(num_pages):
        # API pagination: 'start' tells the API which result to start from
        # Page 0: results 0-9, Page 1: results 10-19, etc.
        params = {
            'engine'    : 'google_jobs',  # Use the Google Jobs search engine
            'q'         : query,
            'api_key'   : api_key,
            'hl'        : 'en',           # Language: English
            'start'     : page * 10       # Pagination offset
        }


        try:
            response = requests.get(SERP_URL, params=params, timeout=15)


            if response.status_code == 200:
                data = response.json()


                # 'jobs_results' is the key in the JSON that holds the job listings
                jobs = data.get('jobs_results', [])


                for job in jobs:
                    # Extract and normalize each job's fields
                    # .get('key', 'default') returns the value if the key exists,
                    # or 'default' if it does not — prevents KeyError crashes
                    all_jobs.append({
                        'title'      : job.get('title', 'Unknown Title'),
                        'company'    : job.get('company_name', 'Unknown Company'),
                        'location'   : job.get('location', 'Unknown Location'),
                        'posted'     : job.get('detected_extensions', {}).get('posted_at', 'Unknown'),
                        'salary'     : job.get('detected_extensions', {}).get('salary', 'Not Disclosed'),
                        'job_type'   : job.get('detected_extensions', {}).get('schedule_type', 'Not Specified'),
                        'description': job.get('description', '')[:300]  # First 300 characters only
                    })


                print(f"  Page {page + 1}: fetched {len(jobs)} jobs")
            else:
                print(f"  Page {page + 1}: API error {response.status_code}")


        except Exception as e:
            print(f"  Page {page + 1}: error — {e}")


    return all_jobs


# ── Actually call the function ────────────────────────────────
job_records = []


if SERP_API_KEY != 'YOUR_SERPAPI_KEY_HERE':
    print(f"Fetching job listings for: '{SEARCH_QUERY}'")
    job_records = fetch_jobs(SEARCH_QUERY, SERP_API_KEY)
    print(f"Total jobs fetched: {len(job_records)}")
else:
    print("No SerpAPI key provided — fallback job data will be loaded next.")





Fetching job listings for: 'Data Engineer India'
  Page 1: fetched 10 jobs
  Page 2: fetched 10 jobs
Total jobs fetched: 20


In [59]:
# ============================================================
# CELL 23 — Fallback Job Data (run if no API key)
# ============================================================


if len(job_records) == 0:
    print("Loading realistic fallback job data...")
    job_records = [
        {'title': 'Data Engineer', 'company': 'Infosys',        'location': 'Bangalore, India',  'posted': '3 days ago',  'salary': 'Rs 12-18 LPA', 'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'TCS',            'location': 'Chennai, India',    'posted': '1 week ago',  'salary': 'Not Disclosed','job_type': 'Full-time'},
        {'title': 'Senior Data Engineer', 'company': 'Wipro',   'location': 'Pune, India',       'posted': '2 days ago',  'salary': 'Rs 20-28 LPA', 'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'Amazon',         'location': 'Hyderabad, India',  'posted': '5 days ago',  'salary': 'Rs 25-35 LPA', 'job_type': 'Full-time'},
        {'title': 'Junior Data Engineer','company': 'HCL',      'location': 'Noida, India',      'posted': '4 days ago',  'salary': 'Rs 6-10 LPA',  'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'Google',         'location': 'Bangalore, India',  'posted': '1 day ago',   'salary': 'Rs 40-60 LPA', 'job_type': 'Full-time'},
        {'title': 'ETL Developer', 'company': 'Cognizant',      'location': 'Hyderabad, India',  'posted': '6 days ago',  'salary': 'Rs 8-14 LPA',  'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'Microsoft',      'location': 'Hyderabad, India',  'posted': '2 days ago',  'salary': 'Rs 30-45 LPA', 'job_type': 'Full-time'},
        {'title': 'Data Pipeline Engineer','company': 'Flipkart','location': 'Bangalore, India', 'posted': '1 week ago',  'salary': 'Rs 22-30 LPA', 'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'Accenture',      'location': 'Mumbai, India',     'posted': '3 days ago',  'salary': 'Rs 10-16 LPA', 'job_type': 'Full-time'},
        {'title': 'Senior Data Engineer','company': 'Amazon',   'location': 'Chennai, India',    'posted': '8 days ago',  'salary': 'Rs 28-40 LPA', 'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'Infosys',        'location': 'Pune, India',       'posted': '2 weeks ago', 'salary': 'Not Disclosed','job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'Swiggy',         'location': 'Bangalore, India',  'posted': '4 days ago',  'salary': 'Rs 18-25 LPA', 'job_type': 'Full-time'},
        {'title': 'ETL Developer', 'company': 'Tech Mahindra',  'location': 'Noida, India',      'posted': '5 days ago',  'salary': 'Rs 7-12 LPA',  'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'Zomato',         'location': 'Delhi, India',      'posted': '3 days ago',  'salary': 'Rs 15-22 LPA', 'job_type': 'Full-time'},
        {'title': 'Senior Data Engineer','company': 'Google',   'location': 'Bangalore, India',  'posted': '1 day ago',   'salary': 'Rs 50-70 LPA', 'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'PayPal',         'location': 'Chennai, India',    'posted': '2 days ago',  'salary': 'Rs 20-32 LPA', 'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'Capgemini',      'location': 'Mumbai, India',     'posted': '1 week ago',  'salary': 'Not Disclosed','job_type': 'Full-time'},
        {'title': 'Junior Data Engineer','company': 'Wipro',    'location': 'Hyderabad, India',  'posted': '3 days ago',  'salary': 'Rs 5-9 LPA',   'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'Oracle',         'location': 'Bangalore, India',  'posted': '4 days ago',  'salary': 'Rs 18-28 LPA', 'job_type': 'Full-time'},
    ]
    print(f"Fallback data loaded: {len(job_records)} job listings")
else:
    print(f"Using live SerpAPI data: {len(job_records)} jobs")




Using live SerpAPI data: 20 jobs


In [60]:
df3 = pd.DataFrame(job_records)
df3.to_excel('job_records.xlsx',index=False)

In [68]:
API_KEY = '59cdc10c3a18e70379de736f2fb45d87'
BASE_URL = 'https://api.openweathermap.org/data/2.5/weather'
CITIES = ['Mumbai', 'Delhi', 'Bangalore', 'Chennai',
          'Hyderabad', 'Kolkata', 'Pune', 'Jaipur']

print(f'API configured for {len(CITIES)} cities')
print(f'Cities: {CITIES}')


API configured for 8 cities
Cities: ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Hyderabad', 'Kolkata', 'Pune', 'Jaipur']

IMPORTANT: Replace YOUR_API_KEY_HERE with your actual key before running.


In [69]:
def fetch_weather_data(city, api_key, unit='metric'):
    """
    Fetches current weather data for a given city from OpenWeatherMap API.

    Parameters:
        city    (str) : The city name (e.g., 'London', 'Mumbai')
        api_key (str) : Your OpenWeatherMap API key
        unit    (str) : Units for temperature ('metric' for Celsius, 'imperial' for Fahrenheit, 'standard' for Kelvin)

    Returns:
        dict : A dictionary containing weather data, or None if an error occurs.
    """
    params = {
        'q': city,
        'appid': api_key,
        'units': unit
    }
    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data for {city}: {e}")
        return None

city_name = 'Mumbai'
if API_KEY != 'YOUR_API_KEY_HERE':
    weather_data = fetch_weather_data(city_name, API_KEY)
    if weather_data:
        print(f"Successfully fetched weather data for {city_name}:")
        print(json.dumps(weather_data, indent=2))
    else:
        print(f"Could not fetch weather data for {city_name}. Please check your API_KEY for validity and authorization.")
else:
    print("API_KEY is not set. Please replace 'YOUR_API_KEY_HERE' in the cell above with your actual key to fetch live data.")
    weather_data = {
        "coord": {"lon": 72.8479, "lat": 19.0144},
        "weather": [{"id": 800, "main": "Clear", "description": "clear sky", "icon": "01d"}],
        "base": "stations",
        "main": {"temp": 30.05, "feels_like": 31.95, "temp_min": 30.05, "temp_max": 30.05, "pressure": 1010, "humidity": 66},
        "visibility": 6000,
        "wind": {"speed": 3.09, "deg": 240},
        "clouds": {"all": 0},
        "dt": 1701345600,
        "sys": {"type": 1, "id": 9052, "country": "IN", "sunrise": 1701318000, "sunset": 1701357600},
        "timezone": 19800,
        "id": 1275339,
        "name": "Mumbai",
        "cod": 200
    }
    print("Using fallback weather data for demonstration:")
    print(json.dumps(weather_data, indent=2))

Successfully fetched weather data for Mumbai:
{
  "coord": {
    "lon": 72.8479,
    "lat": 19.0144
  },
  "weather": [
    {
      "id": 721,
      "main": "Haze",
      "description": "haze",
      "icon": "50d"
    }
  ],
  "base": "stations",
  "main": {
    "temp": 32.99,
    "feels_like": 39.99,
    "temp_min": 32.94,
    "temp_max": 32.99,
    "pressure": 1008,
    "humidity": 62,
    "sea_level": 1008,
    "grnd_level": 1007
  },
  "visibility": 6000,
  "wind": {
    "speed": 6.17,
    "deg": 290
  },
  "clouds": {
    "all": 40
  },
  "dt": 1779966476,
  "sys": {
    "type": 1,
    "id": 9052,
    "country": "IN",
    "sunrise": 1779928268,
    "sunset": 1779975654
  },
  "timezone": 19800,
  "id": 1275339,
  "name": "Mumbai",
  "cod": 200
}


In [75]:
all_cities_weather_data = []

if API_KEY != 'YOUR_API_KEY_HERE':
    for city in CITIES:
        print(f"Fetching weather for {city}...")
        city_data = fetch_weather_data(city, API_KEY)
        if city_data:
            all_cities_weather_data.append(city_data)
        else:
            print(f"Skipping {city} due to fetch error.")

    if all_cities_weather_data:
        processed_all_cities = []
        for data in all_cities_weather_data:
            processed_all_cities.append({
                'city': data.get('name'),
                'country': data.get('sys', {}).get('country'),
                'latitude': data.get('coord', {}).get('lat'),
                'longitude': data.get('coord', {}).get('lon'),
                'temperature_celsius': data.get('main', {}).get('temp'),
                'feels_like_celsius': data.get('main', {}).get('feels_like'),
                'min_temp_celsius': data.get('main', {}).get('temp_min'),
                'max_temp_celsius': data.get('main', {}).get('temp_max'),
                'pressure_hpa': data.get('main', {}).get('pressure'),
                'humidity_percent': data.get('main', {}).get('humidity'),
                'weather_main': data.get('weather', [{}])[0].get('main'),
                'weather_description': data.get('weather', [{}])[0].get('description'),
                'wind_speed_mps': data.get('wind', {}).get('speed'),
                'wind_direction_deg': data.get('wind', {}).get('deg'),
                'cloudiness_percent': data.get('clouds', {}).get('all'),
                'data_timestamp': datetime.fromtimestamp(data.get('dt', 0)).strftime('%Y-%m-%d %H:%M:%S'),
                'sunrise_time': datetime.fromtimestamp(data.get('sys', {}).get('sunrise', 0)).strftime('%H:%M:%S'),
                'sunset_time': datetime.fromtimestamp(data.get('sys', {}).get('sunset', 0)).strftime('%H:%M:%S')
            })

        all_cities_weather_df = pd.DataFrame(processed_all_cities)
        print("\nWeather Data for all cities:")
        display(all_cities_weather_df)
    else:
        print("No weather data fetched for any city.")
else:
    print("API_KEY is not set. Please replace 'YOUR_API_KEY_HERE' in the cell to fetch live data for all cities.")

Fetching weather for Mumbai...
Fetching weather for Delhi...
Fetching weather for Bangalore...
Fetching weather for Chennai...
Fetching weather for Hyderabad...
Fetching weather for Kolkata...
Fetching weather for Pune...
Fetching weather for Jaipur...

Weather Data for all cities:


,city,country,latitude,longitude,temperature_celsius,feels_like_celsius,min_temp_celsius,max_temp_celsius,pressure_hpa,humidity_percent,weather_main,weather_description,wind_speed_mps,wind_direction_deg,cloudiness_percent,data_timestamp,sunrise_time,sunset_time
0,Mumbai,IN,19.0144,72.8479,32.99,39.99,32.94,32.99,1008,62,Haze,haze,6.17,290,40,2026-05-28 11:10:10,00:31:08,13:40:54
1,Delhi,IN,28.6667,77.2167,39.05,44.39,39.05,39.05,1000,37,Haze,haze,7.20,80,20,2026-05-28 11:18:29,23:54:50,13:42:14
2,Bengaluru,IN,12.9762,77.6033,32.58,35.73,30.90,33.05,1008,51,Clouds,scattered clouds,2.06,290,40,2026-05-28 11:16:48,00:22:34,13:11:24
3,Chennai,IN,13.0878,80.2785,34.76,41.76,34.42,35.99,1005,62,Clouds,few clouds,6.17,140,20,2026-05-28 11:18:31,00:11:41,13:00:53
4,Hyderabad,IN,17.3753,78.4744,38.23,39.85,37.73,38.23,1005,30,Clouds,scattered clouds,5.14,320,40,2026-05-28 11:10:59,00:11:32,13:15:28
5,Kolkata,IN,22.5697,88.3697,32.97,39.97,32.97,32.97,1001,66,Haze,haze,3.60,90,40,2026-05-28 11:19:19,23:22:29,12:45:21
6,Pune,IN,18.5196,73.8553,36.89,36.05,36.89,36.89,1007,24,Clouds,few clouds,4.92,292,12,2026-05-28 11:17:28,00:27:59,13:35:59
7,Jaipur,IN,26.9167,75.8167,43.62,42.16,43.62,43.62,999,14,Haze,haze,6.69,250,0,2026-05-28 11:15:44,00:04:06,13:44:10


In [76]:
if 'all_cities_weather_df' in locals() and not all_cities_weather_df.empty:
    print("Cleaned Weather Data DataFrame for all 8 cities:")
    display(all_cities_weather_df)
elif weather_data:
    processed_data = {
        'city': weather_data.get('name'),
        'country': weather_data.get('sys', {}).get('country'),
        'latitude': weather_data.get('coord', {}).get('lat'),
        'longitude': weather_data.get('coord', {}).get('lon'),
        'temperature_celsius': weather_data.get('main', {}).get('temp'),
        'feels_like_celsius': weather_data.get('main', {}).get('feels_like'),
        'min_temp_celsius': weather_data.get('main', {}).get('temp_min'),
        'max_temp_celsius': weather_data.get('main', {}).get('temp_max'),
        'pressure_hpa': weather_data.get('main', {}).get('pressure'),
        'humidity_percent': weather_data.get('main', {}).get('humidity'),
        'weather_main': weather_data.get('weather', [{}])[0].get('main'),
        'weather_description': weather_data.get('weather', [{}])[0].get('description'),
        'wind_speed_mps': weather_data.get('wind', {}).get('speed'),
        'wind_direction_deg': weather_data.get('wind', {}).get('deg'),
        'cloudiness_percent': weather_data.get('clouds', {}).get('all'),
        'data_timestamp': datetime.fromtimestamp(weather_data.get('dt', 0)).strftime('%Y-%m-%d %H:%M:%S'),
        'sunrise_time': datetime.fromtimestamp(weather_data.get('sys', {}).get('sunrise', 0)).strftime('%H:%M:%S'),
        'sunset_time': datetime.fromtimestamp(weather_data.get('sys', {}).get('sunset', 0)).strftime('%H:%M:%S')
    }

    weather_df = pd.DataFrame([processed_data])

    print("Cleaned Weather Data DataFrame (single city):")
    display(weather_df)
else:
    print("No weather data available to process into a DataFrame.")

Cleaned Weather Data DataFrame:


,city,country,latitude,longitude,temperature_celsius,feels_like_celsius,min_temp_celsius,max_temp_celsius,pressure_hpa,humidity_percent,weather_main,weather_description,wind_speed_mps,wind_direction_deg,cloudiness_percent,data_timestamp,sunrise_time,sunset_time
0,Mumbai,IN,19.0144,72.8479,32.99,39.99,32.94,32.99,1008,62,Haze,haze,6.17,290,40,2026-05-28 11:07:56,00:31:08,13:40:54


In [64]:
if 'weather_df' in locals() and not weather_df.empty:
    output_weather_filename = 'cleaned_weather_data.csv'
    weather_df.to_csv(output_weather_filename, index=False)
    print(f"Cleaned weather data saved to {output_weather_filename}")
else:
    print("No cleaned weather data DataFrame found to save.")

Cleaned weather data saved to cleaned_weather_data.csv


---

In [71]:
dummy_salary_data = {
    'Employee ID': [1, 2, 3, 4, 5, 1, 6],
    'Name': ['Alice', 'Bob', 'Charlie', 'Alice', 'David', 'Alice', 'Eve'],
    'Department': ['HR', 'IT', 'Finance', 'HR', 'IT', 'HR', 'Finance'],
    'Monthly Salary': [5000, 6000, 7500, 5000, 6200, 5000, 7000],
    'Bonus': [500, 600, 750, 500, 620, 500, 700]
}
dummy_salary_df = pd.DataFrame(dummy_salary_data);

dummy_excel_filename = 'employee_salary.xlsx'
dummy_salary_df.to_excel(dummy_excel_filename, index=False)
print(f"Dummy Excel file '{dummy_excel_filename}' created.")

print(f"Loading data from '{dummy_excel_filename}'...")
employee_df = pd.read_excel(dummy_excel_filename)

print("Original Employee Salary Data:")
display(employee_df.head())

Dummy Excel file 'employee_salary.xlsx' created.
Loading data from 'employee_salary.xlsx'...
Original Employee Salary Data:


,Employee ID,Name,Department,Monthly Salary,Bonus
0,1,Alice,HR,5000,500
1,2,Bob,IT,6000,600
2,3,Charlie,Finance,7500,750
3,4,Alice,HR,5000,500
4,5,David,IT,6200,620


In [72]:
print(f"Rows before removing duplicates: {len(employee_df)}")
print(f"Duplicate rows found: {employee_df.duplicated().sum()}")

employee_df.drop_duplicates(inplace=True)
employee_df.reset_index(drop=True, inplace=True)

print(f"Rows after removing duplicates: {len(employee_df)}")

employee_df['Yearly Salary'] = (employee_df['Monthly Salary'] * 12) + employee_df['Bonus']

print("Transformed Employee Salary Data (after removing duplicates and calculating yearly salary):")
display(employee_df.head())

Rows before removing duplicates: 7
Duplicate rows found: 1
Rows after removing duplicates: 6
Transformed Employee Salary Data (after removing duplicates and calculating yearly salary):


,Employee ID,Name,Department,Monthly Salary,Bonus,Yearly Salary
0,1,Alice,HR,5000,500,60500
1,2,Bob,IT,6000,600,72600
2,3,Charlie,Finance,7500,750,90750
3,4,Alice,HR,5000,500,60500
4,5,David,IT,6200,620,75020


In [67]:
output_salary_filename = 'cleaned_employee_salary.csv'
employee_df.to_csv(output_salary_filename, index=False)
print(f"Cleaned employee salary data saved to {output_salary_filename}")

Cleaned employee salary data saved to cleaned_employee_salary.csv
